In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler # Z-score

# Limpeza e Pré-processamento

In [10]:
# Carregando a base de dados
df = pd.read_csv('email_phishing_data.csv')

print(f"Tamanho original do dataset: {df.shape}")
df.head()

Tamanho original do dataset: (524846, 9)


,num_words,num_unique_words,num_stopwords,num_links,num_unique_domains,num_email_addresses,num_spelling_errors,num_urgent_keywords,label
0,140,94,52,0,0,0,0,0,0
1,5,5,1,0,0,0,0,0,0
2,34,32,15,0,0,0,0,0,0
3,6,6,2,0,0,0,0,0,0
4,9,9,2,0,0,0,0,0,0


In [11]:
# Removendo duplicatas
df = df.drop_duplicates()
print(f"Tamanho após remover duplicatas: {df.shape}")

Tamanho após remover duplicatas: (205052, 9)


In [17]:
# Removendo features redundantes
colunas_redundantes = ['num_stopwords', 'num_unique_domains'] 
df = df.drop(columns=colunas_redundantes, errors='ignore')
df.head()

,num_words,num_unique_words,num_links,num_email_addresses,num_spelling_errors,num_urgent_keywords,label
0,140,94,0,0,0,0,0
1,5,5,0,0,0,0,0
2,34,32,0,0,0,0,0
3,6,6,0,0,0,0,0
4,9,9,0,0,0,0,0


In [23]:
# Separando features e target
X = df.drop('label', axis=1)
y = df['label']

X.head()

,num_words,num_unique_words,num_links,num_email_addresses,num_spelling_errors,num_urgent_keywords
0,140,94,0,0,0,0
1,5,5,0,0,0,0
2,34,32,0,0,0,0
3,6,6,0,0,0,0
4,9,9,0,0,0,0


Cross-validation e Normalização

In [37]:
# Cross-validation
k = 10
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# Normalização (StandardScaler = z-score)
scaler = StandardScaler()
fold = 1
dados_preparados = []

for train_index, test_index in skf.split(X, y):   
    # Separando os dados de treino e teste para o fold atual
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Normalização
    # (fit_transform) -> Calculando média e desvio padrão SOMENTE no conjunto de treino
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Aplicando a Normalização (transform) no conjunto de teste
    X_test_scaled = scaler.transform(X_test)

    # Guardando os dados prontos para posterior treinamento e avaliação do modelo
    dados_preparados.append({
        'fold': fold,
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test
    })
        
    fold += 1

print(f"Pré-processamento concluído! {len(dados_preparados)} folds estão prontos para uso.")

# Vamos usar a primeira coluna (feature) como exemplo
nome_feature = X.columns[0]

# Exibindo resultado da normalizacão na feature "num_words"
print(f"\n--- Efeito da Normalização na feature: '{nome_feature}' ---")

# Estatísticas ANTES da normalização (usando o X original)
print("\n[ANTES] (Valores brutos da base original):")
print(f"Média         : {X[nome_feature].mean():.2f}")
print(f"Desvio Padrão : {X[nome_feature].std():.2f}")
print(f"Valor Máximo  : {X[nome_feature].max():.2f}") 


# Estatísticas DEPOIS da normalização (usando o Fold 1 preparado)
X_treino_norm = dados_preparados[0]['X_train']
print("\n[DEPOIS] (Valores transformados com Z-Score no Fold 1):")
print(f"Média         : {X_treino_norm[:, 0].mean():.2f}")
print(f"Desvio Padrão : {X_treino_norm[:, 0].std():.2f}")
print(f"Valor Máximo  : {X_treino_norm[:, 0].max():.2f}")


Pré-processamento concluído! 10 folds estão prontos para uso.

--- Efeito da Normalização na feature: 'num_words' ---

[ANTES] (Valores brutos da base original):
Média         : 344.90
Desvio Padrão : 5255.02
Valor Máximo  : 2339682.00

[DEPOIS] (Valores transformados com Z-Score no Fold 1):
Média         : 0.00
Desvio Padrão : 1.00
Valor Máximo  : 422.87
